In [3]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)


import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any


from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase


In [4]:
print(config.llm.groq.available)

['qwen/qwen3.6-27b', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b']


In [5]:
class LLMService:
    def __init__(self, settings, config):
        self.settings = settings
        self.config = config

        self.openrouter_key = settings.OPENROUTER_API_KEY
        self.gemini_key = settings.GEMINI_API_KEY
        self.groq_key = settings.GROQ_API_KEY


    def call_groq(self, model, messages: list):
        from groq import Groq
        client = Groq(api_key=self.settings.GROQ_API_KEY)

        completion = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=self.config.generation.temperature,
            max_completion_tokens=self.config.generation.max_tokens,
            top_p=self.config.generation.top_p,
            # reasoning_effort=self.config.generation.reasoning_effort,
            reasoning_effort="medium", # default / none for qwen 3.6 groq; low / medium / high for gpt groq
            stream=self.config.generation.stream,
            stop=self.config.generation.stop
        )
        return completion

In [6]:
# 1. Đổi tên biến thành chữ thường (llm_service) để tránh lỗi trùng tên Class
llm_service = LLMService(settings, config)

test_messages = [
    # 1. Đặt toàn bộ quy tắc ràng buộc vào đây (Role System luôn đứng đầu)
    {
        "role": "system", 
        "content": (
            "You are a helpful e-commerce assistant. "
            "Always answer entirely in Vietnamese. "
            "Never mix Chinese, Japanese or Korean characters in the final response "
            "unless the user explicitly asks."
        )
    },
    # 2. Tin nhắn của người dùng chỉ chứa câu hỏi thuần túy
    {
        "role": "user", 
        "content": "Xin chào, bạn là ai? Hãy suy nghĩ linh tinh thật dài, khoảng 1000 từ, trả lời thật ngắn để toi test hệ thống."
    }
]

# Gọi API
result = llm_service.call_groq(model=config.llm.groq.available[2], messages=test_messages)


# 3. Sử dụng vòng lặp for để hiển thị chữ chạy thời gian thực (Streaming)
# print("AI trả lời: ", end="")
# for chunk in result:
#     print(chunk.choices[0].delta.content or "", end="")


reasoning = []
content = []

current_mode = None

for chunk in result:
    delta = chunk.choices[0].delta

    if getattr(delta, "reasoning", None):

        reasoning.append(delta.reasoning)

        if current_mode != "reasoning":
            print("\n===== THINKING =====")
            current_mode = "reasoning"

        print(delta.reasoning, end="", flush=True)

    elif getattr(delta, "content", None):

        content.append(delta.content)

        if current_mode != "content":
            print("\n===== ANSWER =====")
            current_mode = "content"

        print(delta.content, end="", flush=True)


===== THINKING =====
The user wants an answer in Vietnamese, with no Chinese, Japanese, Korean. The user asks: "Xin chào, bạn là ai? Hãy suy nghĩ linh tinh thật dài, khoảng 1000 từ, trả lời thật ngắn để toi test hệ thống." There's a contradiction: "suy nghĩ linh tinh thật dài, khoảng 1000 từ" but "trả lời thật ngắn". They want a long think but answer short? The user says "to test system". The system instructions: "Always answer entirely in Vietnamese. Never mix Chinese, Japanese or Korean characters in the final response unless the user explicitly asks." So we should answer in Vietnamese. They want a long think, but answer short. Probably they want the assistant to show it's able to think in a long way but only give short answer. But the system says "Always answer entirely in Vietnamese". So answer in Vietnamese. The user wants to test system: we should comply. Provide a short answer but maybe with the thought process? The system says we can do internal analysis. But the final answer 